In [1]:
from google.colab import drive
from pathlib import Path
import sys
!pip install datasets transformers huggingface_hub
from datasets import load_dataset

drive.mount('/content/drive/')
strPath = "drive/MyDrive/Spurious_Correlations/emnlp-2020-spurious/"
# sys.path.insert(0,"drive/MyDrive/Spurious_Correlations/Identifying-and-Mitigating-Spurious-Correlations-for-Improving-Robustness-in-NLP-Models")
filePath = Path(strPath)
%cd $filePath

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 468.7/468.7 kB 9.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 40.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.1/200.1 kB 15.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.9/132.9 kB 12.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.2/212.2 kB 9.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 10.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 10.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 15.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 1.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.8/158.8 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.6/264.6 kB 4.7 MB/s eta 0:00:00
Mounted at /cont

#### [Get data and top words](#data)
- Get data from file and construct DataSet object
- Get top words, placebo words

#### [Edit sentences and get embeddings for edited sentences](#sentence_edit)
- Edit sentences by removing top / placebo / empty words
- Sentence embedding by concatinating last 4 layers of BERT embeddings

#### [Matching sentences and calculate ITE:](ite_match)
- Treatment match
- Placebo match
- Control match
- E.g., "This is a good movie" matched with "This is a bad movie"

In [2]:
import pickle
import io, time
from io import BytesIO
from IPython.display import display
import pickle, tarfile, random, re, requests
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
pd.set_option('max_colwidth', -1)

import sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression

import torch
from transformers import * # here introduces bert
from tqdm.notebook import tqdm

import warnings
warnings.filterwarnings("ignore")

# from allennlp.predictors.predictor import Predictor

<ipython-input-2-03590407ffc1>:9: FutureWarning: Passing a negative integer is deprecated in version 1.0 and will not be supported in future version. Instead, use None to not limit the column width.
  pd.set_option('max_colwidth', -1)
/usr/local/lib/python3.9/dist-packages/transformers/generation_utils.py:24: FutureWarning: Importing `GenerationMixin` from `src/transformers/generation_utils.py` is deprecated and will be removed in Transformers v5. Import as `from transformers import GenerationMixin` instead.
  warnings.warn(
/usr/local/lib/python3.9/dist-packages/transformers/generation_tf_utils.py:24: FutureWarning: Importing `TFGenerationMixin` from `src/transformers/generation_tf_utils.py` is deprecated and will be removed in Transformers v5. Import as `from transformers import TFGenerationMixin` instead.
  warnings.warn(
/usr/local/lib/python3.9/dist-packages/transformers/generation_flax_utils.py:24: FutureWarning: Importing `FlaxGenerationMixin` from `src/transformers/generation_f

In [3]:
from data_structure import Dataset, SentenceEdit, get_IMDB, get_kindle, get_toxic_comment, get_toxic_tw, get_sst2, get_yelp

#### Get data and top words <a id='data'></a>

In [4]:
def simple_vectorize(df):
    """
    Vectorize text
    """
    vec = CountVectorizer(min_df=5, binary=True, max_df=.8)
    # print("Investigating df")
    # print(df)
    X = vec.fit_transform(df.text)
    print(X.shape)
    y = df.label.values
    #feats = np.array(vec.get_feature_names())
    feats = np.array(vec.get_feature_names_out())
    print(len(feats))
    
    return X, y, vec, feats

In [5]:
def print_coef(clf, feats, n=10, pattern=None, feature_counts=None):
    """
    sort and print words by coef stregth (abs(coef))
    """
    list_of_top_n_both_classes = []
    if len(clf.classes_) == 2:
        coefs = [-1*clf.coef_[0], clf.coef_[0]] # change the coef relation corresponding with each class
    else:
        coefs = clf.coef_

    print("coefs look like: ", len(coefs[0]))
    for label, coef in zip(clf.classes_, coefs):
        print("\nTop features for class %s" % str(label))
        if pattern:
            # restrict to features matching pattern
            coef = coef.copy()
            coef[[i for i,s in enumerate(feats) if len(re.findall(pattern, s)) == 0]] = 0
        
        # topi = coef.argsort()[::-1][:n]
        topi = coef.argsort()[::-1]# don't use top n, use all
        # print(topi)
        # s = ' '.join('%s/%.2f' % (f,c) for f, c in zip(feats[topi], coef[topi]))
        topi_counts = [feature_counts[feats[i]] for i in topi]
        list_of_top_n = [ (f,c, count) for f, c, count in zip(feats[topi], coef[topi], topi_counts)]
        #print(s)
        print(list_of_top_n[:10])
        print(len(list_of_top_n))
        list_of_top_n_both_classes.append(list_of_top_n)
    return list_of_top_n_both_classes
        
def get_top_terms(dataset, coef_thresh=.5, placebo_thresh=.5, C=1, feature_counts=None):
    """
    Fit classifier, print top-n terms;
    Top features (features have high coef): abs(coef) >= thresh
    Placebos (features have low coef): abs(coef) <= thresh
    """
    clf = LogisticRegression(class_weight='balanced', C=C, solver='lbfgs', max_iter=1000)
    clf.fit(dataset.X, dataset.y)
    
    list_of_top_n_both_classes = print_coef(clf, dataset.feats, n=100, feature_counts=feature_counts)
    #print('dummy coef= %.3f' % clf.coef_[0][dataset.vec.vocabulary_[DUMMY_TERM]])
    
    # print("mean of clf_coeff: ", np.mean(abs(clf.coef_[0])))
    # top_feature_idx = np.where(abs(clf.coef_[0]) >= coef_thresh)[0]             # Just use the whole thing as the top features. Don't limit
    # placebo_feature_idx = np.where(abs(clf.coef_[0]) <= placebo_thresh)[0]
    # feature_coef = np.array([float("%.3f" % c) for c in clf.coef_[0]]) 
    
    # return top_feature_idx, placebo_feature_idx, feature_coef
    return list_of_top_n_both_classes

In [25]:
def get_dataset_embeddings():
    """
    1. Get data from file and construct DataSet object
    2. Get top words, placebo words
    3. Edit sentences and get embeddings for edited sentences:
        Remove top words;
        Remove placebo words;
        Original sentences without edit;
    """
    random.seed(42)
    datasets = []
    for get_data_df, moniker, coef_thresh, placebo_thresh in [
            (get_IMDB, 'imdb', 1.0, 0.1),
            (get_sst2, 'sst2', 1.0, 0.1),
            (get_yelp, 'yelp', 0.5, 0.1),
           # (get_kindle, 'kindle', 0.9, 0.2),
            #(get_toxic_comment, 'toxic', 1.0, 0.05), 
            #(get_toxic_tw, 'toxic_tw', 0.7, 0.2),
        ]: 
        
        # Get data and show basic information
        df = get_data_df()
        #print(df)
        X, y, vec, feats = simple_vectorize(df) # vectorize text
        ds = Dataset(X, y, vec, df, moniker) # construct dataset object

        #print(vec.get_feature_names_out())
        features = vec.get_feature_names_out()
        counts = X.toarray().sum(axis=0)
        feature_counts = dict([(token,count) for token,count in zip(features,counts)])
        
        # print("count for a specific word: ", feature_counts['impeccable'])
        print('%s dataset, %d instances' % (moniker,len(df)))
        print('Label distribution: %s' % str(Counter(y).items()))
        print('Feature matrix: %s' % str(X.shape))
        
        # Get top features 
        # ds.top_feature_idx, ds.placebo_feature_idx, ds.coef = get_top_terms(ds, coef_thresh=coef_thresh, placebo_thresh=placebo_thresh, C=1, feature_counts=feature_counts)
        list_of_top_n_both_classes = get_top_terms(ds, coef_thresh=coef_thresh, placebo_thresh=placebo_thresh, C=1, feature_counts=feature_counts)
        print("top 5 words from both classes: ", list_of_top_n_both_classes[0][-1])
        # print("Obtained top features idx")
        # ds.top_features = feats[ds.top_feature_idx]
        print("Obtained top features")
        # ds.placebo_features = feats[ds.placebo_feature_idx]
        # print('\n%d top terms: %d pos, %d neg\n' % (len(ds.top_features), len(np.where(ds.coef[ds.top_feature_idx]>0)[0]), len(np.where(ds.coef[ds.top_feature_idx]<0)[0])))
        # print('\n%d placebo terms: %d pos, %d neg\n' % (len(ds.placebo_features), len(np.where(ds.coef[ds.placebo_feature_idx]>0)[0]), len(np.where(ds.coef[ds.placebo_feature_idx]<0)[0])))

        if moniker == 'sst2':
          # Save the sst2 top terms and scores
          with open('tokens/sst2_important_tokens.csv', 'w') as fp:
            fp.write('\n'.join('%s,%s,%s' % x for x in list_of_top_n_both_classes[0]))

        elif moniker == 'yelp':
          with open('tokens/yelp_important_tokens.csv', 'w') as fp:
            fp.write('\n'.join('%s,%s,%s' % x for x in list_of_top_n_both_classes[0]))
        elif moniker == 'imdb':
          with open('tokens/imdb_important_tokens.csv', 'w') as fp:
            fp.write('\n'.join('%s,%s,%s' % x for x in list_of_top_n_both_classes[0]))

          # Save the yelp top terms and scores
        
        datasets.append(ds) 
    return datasets
# pickle.dump(datasets, open('/data/zwang/2020_S/Toxic/Concat_last4_emb/data_with_placebo_match/datasets_emb.pickle','wb'))


In [26]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

start = time.time()
datasets = get_dataset_embeddings()
end = time.time()
print((end-start)/60)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
(10662, 4574)
4574
new dataset with 10662 records


,label,text
0,-1,"simplistic , silly and tedious ."


imdb dataset, 10662 instances
Label distribution: dict_items([(-1, 5331), (1, 5331)])
Feature matrix: (10662, 4574)
coefs look like:  4574

Top features for class -1
[('boring', 2.151938541382763, 54), ('bore', 2.1148557651365953, 16), ('dull', 2.065038117171978, 79), ('supposed', 2.000458767304094, 25), ('fails', 1.9603801384161195, 49), ('badly', 1.8892719044566748, 25), ('unless', 1.78556801491156, 16), ('routine', 1.773892509723992, 25), ('waste', 1.7693549404896123, 26), ('mindless', 1.7595036292476192, 18)]
4574

Top features for class 1
[('unexpected', 2.0251472453775015, 25), ('engrossing', 2.015944276321345, 33), ('powerful', 1.9250075441283714, 51), ('count', 1.8694406207489953, 17), ('entertain', 1.8362407929361533, 12), ('enjoyable', 1.8186605665148667, 61), ('provides', 1.8117961857814122, 30), ('wonderful', 1.7969580369666682, 34), ('smarter', 1.7616936694417706, 12), ('skin', 1.7445378751645328, 23)]
4574
top 5 words from both classes:  ('unexpected', -2.0251472453775015

  0%|          | 0/3 [00:00<?, ?it/s]

(67349, 10597)
10597
new dataset with 67349 records


...

sst2 dataset, 67349 instances
Label distribution: dict_items([(0, 29780), (1, 37569)])
Feature matrix: (67349, 10597)
coefs look like:  10597

Top features for class 0
[('lacking', 4.352772202986616, 106), ('lacks', 4.223907826419257, 134), ('worst', 4.076367148490139, 270), ('failure', 3.796342395840013, 83), ('mess', 3.636636006013499, 149), ('devoid', 3.604055853783416, 73), ('stupid', 3.5293669884516072, 142), ('bore', 3.288809561531146, 62), ('squanders', 3.2568028883628433, 23), ('waste', 3.2007272046973565, 114)]
10597

Top features for class 1
[('remarkable', 3.6478692603342995, 176), ('refreshing', 3.427486610344488, 77), ('powerful', 3.3183135901933, 242), ('wonderful', 3.2206448475901417, 174), ('hilarious', 3.1655582717788935, 192), ('beautiful', 3.15767580771336, 251), ('treat', 3.005624975770742, 75), ('fascinating', 3.0014715199744084, 304), ('enjoyable', 3.001078432930215, 266), ('terrific', 2.9553578076101044, 156)]
10597
top 5 words from both classes:  ('remarkable', 

  0%|          | 0/2 [00:00<?, ?it/s]

(38000, 18183)
18183
new dataset with 38000 records


,text,label
0,"Contrary to other reviews, I have zero complaints about the service or the prices. I have been getting tire service here for the past 5 years now, and compared to my experience with places like Pep Boys, these guys are experienced and know what they're doing. \nAlso, this is one place that I do not feel like I am being taken advantage of, just because of my gender. Other auto mechanics have been notorious for capitalizing on my ignorance of cars, and have sucked my bank account dry. But here, my service and road coverage has all been well explained - and let up to me to decide. \nAnd they just renovated the waiting room. It looks a lot better than it did in previous years.",1


yelp dataset, 38000 instances
Label distribution: dict_items([(1, 19000), (0, 19000)])
Feature matrix: (38000, 18183)
coefs look like:  18183

Top features for class 0
[('worst', 3.1984819292443527, 1651), ('disappointment', 3.1177600005736053, 442), ('bland', 2.9117694545841397, 1080), ('mediocre', 2.8705895524302485, 739), ('horrible', 2.5348617701329856, 1463), ('meh', 2.399400011926845, 360), ('lacked', 2.3542593763841735, 203), ('terrible', 2.3520044300095795, 1269), ('lacks', 2.33686770038435, 74), ('tasteless', 2.3002965988705104, 334)]
18183

Top features for class 1
[('delicious', 2.186613152217476, 3197), ('fantastic', 2.1525430640058025, 1037), ('amazing', 2.0727960027766894, 2810), ('downside', 2.057155164320224, 140), ('excellent', 2.046159643175952, 1753), ('awesome', 1.9724095875364955, 2357), ('refreshing', 1.9384570443648865, 236), ('helps', 1.8696762713968047, 98), ('perfect', 1.8290674898056836, 1503), ('outstanding', 1.7641283904837648, 396)]
18183
top 5 words from 

In [ ]:
# pickle.dump(datasets, open('/data/zwang/2020_S/EMNLP/V_7_rerun/datasets_emb.pickle','wb'))

### Matching sentences and calculate ITE <a id='ite_match'></a>

- Treatment match:  context_A + top_termA = context_B + top_term_B <br>

- Placebo match:  context_A + top_termA = context_B + non_top_term_B <br>

- Control match:  context_A + top_termA = sentence_B (not contain top_termA) <br>

- Takes 6 hours to run one matching strategy for all datasets (one time run and save for future use)

In [ ]:
def find_matched_sentence(t_sentObj, c_sentObj_list, min_sim=.7):
    """
    For each treatment sentence (sentence contain a top word), find a matched sentence (sentences with similar contexts);
        
    Find the sentence with closest contexts:
    context_A = sentence_A - word_A
    context_B = sentence_B - word_B
    
    for context_A:
        sort similarity score of (context_A, all other contexts) in descending order
        if((sentence_A != sentence_B) and (word_A != word_B) and (cos(context_A,context_B)>0.7)):
            context_B is a match for context_A
    
    diff: difference between sentence_A.label - sentence_B.label
    """
    
    # similarity between current treatment context with all other contexts
    sims = cosine_similarity([t_sentObj.embedding],[c_sent.embedding for c_sent in c_sentObj_list])[0]
    
    match = None
    match_sim = 0
    for c_sentObj, sim in sorted(zip(c_sentObj_list, sims), key=lambda x: -x[1]): # find the first most similar match
        if((sim >= min_sim) and (c_sentObj.sentence_idx != t_sentObj.sentence_idx) and (c_sentObj.remove_wd != t_sentObj.remove_wd) and (not re.search(r'(?i)\b%s\b' % t_sentObj.remove_wd, c_sentObj.context))):
            match = c_sentObj
            match_sim = sim        
            break
            
    if match:
        diff = (t_sentObj.label - match.label)
    else:
        print('no match')
        diff = 0
#         diff = np.nan

    return diff, match_sim, match # [(match_sim, target_sentence)], [(match_sim,match)]


In [ ]:
def calculate_ites_from_matched_sentences(ds_data, matchby='treat', min_sim=0.01):
    """
    For each treatment sentence (sentence contain a top word), find a matched sentence (sentences with similar contexts);
    
    t_sentObj_list: a list of SentenceEdit objects for sentences with top words removed
    c_sentObj_list: a list of SentenceEdit objects for sentences with top / placebo / '' words removed
    matchby = treat / control / placebo
    """
    t_sentObj_list = ds_data.topwd_sentObj_list
    
    if(matchby == 'treat'):
        c_sentObj_list = ds_data.topwd_sentObj_list
    elif(matchby == 'placebo'):
        c_sentObj_list = ds_data.placebowd_sentObj_list
    elif(matchby == 'control'):
        c_sentObj_list = ds_data.original_sentObj_list
    
    matched_pairs = []    
    for t_sentObj in tqdm(t_sentObj_list):
        ite_bylabel, sim, control_obj = find_matched_sentence(t_sentObj, c_sentObj_list, min_sim=min_sim)
            
        matched_pairs.append(
            {
                'term': t_sentObj.remove_wd,
                'sentence_id': t_sentObj.sentence_idx,
                'treat_obj': t_sentObj,
                'control_obj': control_obj, # this is an object             
                'similarity': sim,
                'difference': t_sentObj.embedding - control_obj.embedding, 
                'ite': ite_bylabel,
            }
        )
    return pd.DataFrame(matched_pairs)

#### Experiments using treatment match

In [ ]:
# ds_imdb, ds_kindle, ds_toxic, ds_toxic_tw = datasets
ds_imdb, ds_kindle, ds_toxic, ds_toxic_tw = pickle.load(open('/data/zwang/2020_S/EMNLP/V_7_rerun//datasets_emb.pickle','rb'))
datasets = [ds_imdb, ds_kindle, ds_toxic, ds_toxic_tw]

 
start = time.time()
data_ites_byTreat = []
for ds_data in datasets:
    ds_data.ites = calculate_ites_from_matched_sentences(ds_data, matchby='treat', min_sim=0.01)
    data_ites_byTreat.append(ds_data)
    
end = time.time()
print((end-start)/60)
pickle.dump(data_ites_byTreat, open('/data/zwang/2020_S/EMNLP/V_7_rerun/datasets_treat_match.pickle', 'wb'))

IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)

IOPub message rate exceed

IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)




1293.4846917788188


#### Experiments using placebo match

In [ ]:
# ds_imdb, ds_kindle, ds_toxic, ds_toxic_tw = pickle.load(open('/data/zwang/2020_S/Toxic/Concat_last4_emb/V_1_treat/datasets_emb_mindf5.pickle','rb'))
# datasets = [ds_imdb, ds_kindle, ds_toxic, ds_toxic_tw]

start = time.time()

data_ites_byPlacebo = []
for ds_data in datasets:
    ds_data.ites = calculate_ites_from_matched_sentences(ds_data, matchby='placebo', min_sim=0.01)
    data_ites_byPlacebo.append(ds_data)
    
end = time.time()
print((end-start)/60)
pickle.dump(data_ites_byPlacebo, open('/data/zwang/2020_S/Toxic/Concat_last4_emb/V_3_placebo/datasets_placebo_match.pickle', 'wb'))


#### Experiments using control match

In [ ]:
# ds_imdb, ds_kindle, ds_toxic, ds_toxic_tw = pickle.load(open('/data/zwang/2020_S/Toxic/Concat_last4_emb/V_1_treat/datasets_emb_mindf5.pickle','rb'))
# datasets = [ds_imdb, ds_kindle, ds_toxic, ds_toxic_tw]

start = time.time()

data_ites_byControl = []
for ds_data in datasets:
    ds_data.ites = calculate_ites_from_matched_sentences(ds_data, matchby='control', min_sim=0.01)
    data_ites_byControl.append(ds_data)

end = time.time()
print((end-start)/60)
pickle.dump(data_ites_byControl, open('/data/zwang/2020_S/Toxic/Concat_last4_emb/V_2_control/datasets_control_match.pickle', 'wb'))


#### Manually check label correctness for Kindle
- Sentence labels are inherited from document labels

In [ ]:
df_kindle = get_kindle()
df_kindle.shape

(20233, 3)

In [ ]:
df_kindle.head()

,text,rating,label
0,This was a very fun story,5,1
1,Not fast moving but a very well managed pace,5,1
2,The story line is an interesting take on zombie mythology and is a great journey,5,1
3,"The story was good, but I was getting very irritated at all the grammatical and spelling errors",2,-1
4,series is always a good read,5,1


In [ ]:
df_kindle.iloc[3648].text

"This was a teaser, I can't wait for part 2"

In [ ]:
rand_idx = random.sample(list(df_kindle.index),200)
df_rand = df_kindle.iloc[rand_idx]
df_rand

,text,rating,label
3648,So I do not want to sway them either way,1,-1
819,I thought this was a sweet story,4,1
9012,"my favorite book is Deed of Paksenarrion, which takes me a day or two to read",1,-1
8024,"I think the author had fun writing this book, but it's the sort of food I throw together without the benefit of a recipe",2,-1
7314,You are supposed to laugh here if you loved this,1,-1
...,...,...,...
7216,I did finish it but would not reccomend it,2,-1
235,However it was just too confusing,2,-1
2326,"Funny, sexy and a whole lot of fun from start to finish",5,1
1929,figured why not give the story a try,2,-1


In [ ]:
# df_rand.to_csv('/data/zwang/2020_S/EMNLP/kindle_random_samples.csv')

In [ ]:
df_rand.head()

,text,rating,label
17911,"Apart from that, there isn't much to the plot",4,1
12430,A hot steamy love affair with secrets felonies and hotties,5,1
1319,"Corrupt cop, porn girlfriend, porn producer = predictable crap triangle",2,-1
16636,The book started to get interesting and then it ended and is over,2,-1
4264,Not much depth to it,2,-1


In [ ]:
df_rand_labeled = pd.read_csv('/data/zwang/2020_S/EMNLP/kindle_random_samples.csv')
df_rand_labeled.head()

,Unnamed: 0,Zhao,text,rating,label
0,17911,-1,"Apart from that, there isn't much to the plot",4,1
1,12430,1,A hot steamy love affair with secrets felonies and hotties,5,1
2,1319,-1,"Corrupt cop, porn girlfriend, porn producer = predictable crap triangle",2,-1
3,16636,-1,The book started to get interesting and then it ended and is over,2,-1
4,4264,-1,Not much depth to it,2,-1


In [ ]:
df_rand_labeled['Unnamed: 0'].values

array([17911, 12430,  1319, 16636,  4264])

In [ ]:
df_rand_labeled[df_rand_labeled['Zhao'] == df_rand_labeled['label']].shape

(484, 5)

In [ ]:
193/200, 16/500, 484/500

(0.965, 0.032, 0.968)

In [ ]:
rand_idx_2 = random.sample(list(set(df_kindle.index) - set(df_rand_labeled['Unnamed: 0'].values)),300)
len(rand_idx_2)

300

In [ ]:
set(rand_idx_2).intersection(set(df_rand_labeled['Unnamed: 0'].values))

set()

In [ ]:
df_rand_2 = df_kindle.iloc[rand_idx_2]
df_rand_2.to_csv('/data/zwang/2020_S/EMNLP/kindle_random_samples_2.csv')

#### Out-of-vocabulary words in each train / test dataset

In [ ]:
ds = pickle.load(open('/data/zwang/2020_S/EMNLP/V_7_rerun/datasets_emb.pickle','rb'))
ds_imdb = ds[0]
ds_kindle = ds[1]
ds_toxic = ds[2]
ds_tw = ds[3]
ds_kindle_short = pickle.load(open('/data/zwang/2020_S/EMNLP/V_6_shortSents/kindle_emb.pickle','rb'))
ds_toxic_short = pickle.load(open('/data/zwang/2020_S/EMNLP/V_6_shortSents/toxic_emb.pickle','rb'))

In [ ]:
ds_imdb.df.shape, len(ds_imdb.top_features)

((10662, 3), 366)

In [ ]:
ds_kindle_short.df.shape, len(ds_kindle_short.top_features)

((20233, 4), 270)

In [ ]:
len(set(ds_imdb.top_features).intersection(set(ds_kindle_short.top_features)))

46

In [ ]:
46/366

0.12568306010928962

In [ ]:
ds_toxic_short.df.shape, len(ds_toxic_short.top_features)

((15216, 5), 329)

In [ ]:
ds_tw.df.shape, len(ds_tw.top_features)

((6774, 4), 341)

In [ ]:
len(set(ds_toxic_short.top_features).intersection(set(ds_tw.top_features)))

29

In [ ]:
29/341

0.08504398826979472